# Vamos encher o carrinho: o que os pedidos de um app de mercado revelam sobre hábito de compra

## Introdução

Analisei dados da Instacart, um app de entrega de supermercado parecido com o iFood Mercado. São cerca de **479 mil pedidos** e **4,5 milhões de itens** comprados.

Perguntas de negócio que guiaram a análise:
- **Quando** as pessoas compram? (hora e dia da semana)
- **De quanto em quanto tempo** elas voltam?
- **O que** compram, o que recompram e o que colocam primeiro no carrinho?

Esse tipo de resposta orienta horário de campanha, régua de reativação, estoque e promoção.

A base veio com problemas (valores ausentes e duplicados), então antes de qualquer conclusão fiz a limpeza.

### Como fiz
1. **Visão geral:** carregar as 5 tabelas e entender a estrutura.
2. **Preparação:** duplicados, valores ausentes e tipos de dados, investigando a causa de cada problema.
3. **Análise:** padrões de horário, recorrência, produtos e clientes.

## Dicionário de dados

São cinco tabelas, ligadas por IDs:

- `instacart_orders.csv`: cada linha corresponde a um pedido no aplicativo da Instacart
    - `'order_id'`: é o número que identifica cada pedido de forma exclusiva
    - `'user_id'`: é o número de identificação exclusivo da conta de cada cliente
    - `'order_number'`: é o número de vezes que o cliente fez um pedido
    - `'order_dow'`: é o dia da semana em que o pedido foi feito (0 é domingo)
    - `'order_hour_of_day'`: é a hora do dia em que o pedido foi feito
    - `'days_since_prior_order'`: é o número de dias desde que o cliente fez seu pedido anterior




- `products.csv`: cada linha corresponde a um produto exclusivo que os clientes podem comprar
    - `'product_id'`: é o número de identificação unívoco de cada produto
    - `'product_name'`: é o nome do produto
    - `'aisle_id'`: é o número de identificação exclusivo de cada categoria de corredor do supermercado
    - `'department_id'`: é o número de identificação exclusivo de cada categoria de departamento do supermercado




-	`order_products.csv`: cada linha corresponde a um item incluído em um pedido
    -	`'order_id'`: é o número que identifica cada pedido de forma exclusiva
    -	`'product_id'`: é o número de identificação exclusivo de cada produto
    -	`'add_to_cart_order'`: é a ordem sequencial em que cada item foi colocado no carrinho
    -	`'reordered'`: 0 se o cliente nunca comprou o produto antes, 1 se já o comprou




-	`aisles.csv`
    -	`'aisle_id'`: é o número de identificação exclusivo de cada categoria de corredor do supermercado
    -	`'aisle'`: é o nome do corredor



-	`departments.csv`
    -	`'department_id'`: é o número de identificação exclusivo de cada categoria de departamento do supermercado
    -	`'department'`: é o nome do departamento


# Etapa 1. Visão geral dos dados

Carreguei as cinco tabelas. Os arquivos usam `;` como separador, então indiquei isso no `read_csv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
instacart_orders = pd.read_csv('instacart_orders.csv', sep=';')
instacart_orders.head()

In [ ]:
products = pd.read_csv('products.csv', sep=';')
products.head()

In [ ]:
aisles = pd.read_csv('aisles.csv', sep=';')
aisles.head()

In [ ]:
departments = pd.read_csv('departments.csv', sep=';')
departments.head()

In [ ]:
order_products = pd.read_csv('order_products.csv', sep=';')
order_products.head()

In [ ]:
# Estrutura de cada tabela
tabelas = {
    'instacart_orders': instacart_orders,
    'products': products,
    'aisles': aisles,
    'departments': departments,
    'order_products': order_products,
}

for nome, tabela in tabelas.items():
    print(f'===== {nome}')
    tabela.info(show_counts=True)
    print()

### O que a visão geral mostrou

| Tabela | Linhas | O que é |
|---|---|---|
| `order_products` | 4.545.007 | itens dentro de cada pedido |
| `instacart_orders` | 478.967 | pedidos |
| `products` | 49.694 | produtos |
| `aisles` | 134 | corredores |
| `departments` | 21 | departamentos |

Pontos de atenção para a limpeza:
- Ausentes em `product_name` (1.258) e `days_since_prior_order` (28.819).
- `add_to_cart_order` está como `float`, mas deveria ser inteiro. Isso costuma ser sinal de valores ausentes na coluna.
- As tabelas se ligam por `order_id`, `product_id`, `aisle_id` e `department_id`, o que permite cruzar tudo.

# Etapa 2. Preparação dos dados

Três frentes: duplicados, valores ausentes e tipos de dados. Em cada caso, antes de corrigir, procurei entender **por que** o problema existe. É isso que define se o certo é remover, preencher ou manter.

## Valores duplicados

### Pedidos (`instacart_orders`)

In [ ]:
# Linhas repetidas (mostrando as duas ocorrências de cada uma)
print('Pedidos duplicados:', instacart_orders.duplicated().sum())
instacart_orders[instacart_orders.duplicated(keep=False)].sort_values('order_id')

Há **15 pedidos duplicados** (30 linhas, contando as duas ocorrências de cada um). E todos têm algo em comum: foram feitos **na quarta-feira, às 2h da manhã** (`order_dow = 3` e `order_hour_of_day = 2`).

Coincidência demais. Fui olhar todos os pedidos desse horário:

In [ ]:
# Todos os pedidos feitos na quarta-feira às 2h
wednesday_2am = instacart_orders[
    (instacart_orders['order_dow'] == 3) &
    (instacart_orders['order_hour_of_day'] == 2)
]
print('Linhas na quarta às 2h:', len(wednesday_2am))

São 121 linhas na quarta às 2h, e 30 delas são duplicatas, cerca de 1 em cada 4. Num horário de pouquíssimo movimento, isso aponta para uma **falha de sistema** (registro gravado duas vezes), e não para clientes repetindo o pedido.

Como são cópias exatas, removi:

In [ ]:
# Removendo os pedidos duplicados
instacart_orders = instacart_orders.drop_duplicates().reset_index(drop=True)

In [ ]:
# Conferindo: linhas duplicadas
print(instacart_orders.duplicated().sum())

In [ ]:
# Conferindo: IDs de pedido duplicados
print(instacart_orders['order_id'].duplicated().sum())

Zero duplicados, tanto na linha inteira quanto no `order_id`. Agora cada pedido aparece uma única vez.

### Produtos (`products`)

In [ ]:
# Linhas inteiras duplicadas
print(products.duplicated().sum())

In [ ]:
# IDs de produto duplicados
print(products['product_id'].duplicated().sum())

Sem linhas ou IDs repetidos. Mas o mesmo nome pode aparecer com maiúsculas diferentes ("Of" x "of"). Por isso, padronizei tudo em minúsculas antes de comparar:

In [ ]:
# Padronizando os nomes em minúsculas
products['product_name'] = products['product_name'].str.lower()

In [ ]:
# Nomes repetidos, contando os vazios
print(products['product_name'].duplicated().sum())

Esse 1.361 está inflado: os 1.258 nomes vazios contam como repetidos entre si. Tirando os vazios:

In [ ]:
# Só os nomes preenchidos
nomes = products['product_name'].dropna()

In [ ]:
# Nomes repetidos, sem os vazios
print(nomes.duplicated().sum())

In [ ]:
# Quais nomes se repetem
print(nomes[nomes.duplicated(keep=False)].sort_values().unique())

**104 nomes de produto se repetem**, como "aged balsamic vinegar of modena".

**Decisão: mantive todos.** Cada um tem um `product_id` diferente, então podem ser produtos distintos com o mesmo nome. Apagar essas linhas distorceria as contagens de vendas da Etapa 3.

### Departamentos (`departments`)

In [ ]:
# Linhas inteiras duplicadas
print(departments.duplicated().sum())

In [ ]:
# IDs de departamento duplicados
print(departments['department_id'].duplicated().sum())

Sem duplicados. Nada a corrigir.

### Corredores (`aisles`)

In [ ]:
# Linhas inteiras duplicadas
print(aisles.duplicated().sum())

In [ ]:
# IDs de corredor duplicados
print(aisles['aisle_id'].duplicated().sum())

Sem duplicados. Nada a corrigir.

### Itens dos pedidos (`order_products`)

In [ ]:
# Linhas inteiras duplicadas
print(order_products.duplicated().sum())

Também verifiquei um caso que a checagem de linha inteira não pega: o **mesmo produto lançado duas vezes no mesmo pedido**.

In [ ]:
# Mesmo produto lançado duas vezes no mesmo pedido
print(order_products.duplicated(subset=['order_id', 'product_id']).sum())

Nenhum duplicado nos dois níveis. A tabela está limpa.

## Valores ausentes

Três colunas têm ausentes: `product_name`, `days_since_prior_order` e `add_to_cart_order`. Investiguei cada uma antes de decidir o tratamento.

### Nomes de produtos ausentes

In [ ]:
# Produtos sem nome
print(products['product_name'].isna().sum())

São 1.258 produtos sem nome. Antes de preencher, quis saber se existe um padrão por trás:

In [ ]:
# Em quais corredores estão os produtos sem nome?
products[products['product_name'].isna()]['aisle_id'].unique()

Todos estão no **corredor 100**.

In [ ]:
# Todos estão no departamento 21?
(products.loc[products['product_name'].isna(), 'department_id'] == 21).all()

E todos no **departamento 21**. O que são esse corredor e esse departamento?

In [ ]:
# O que são o corredor 100 e o departamento 21?
print(aisles[aisles['aisle_id'] == 100])
print()
print(departments[departments['department_id'] == 21])

Os dois se chamam **"missing"** na própria base. Ou seja, não é falha aleatória: são produtos cadastrados sem categoria e sem nome.

Não tem como recuperar esses nomes, mas os produtos têm vendas reais. Por isso **preenchi com "Unknown"** em vez de apagar.

In [ ]:
# Preenchendo os nomes ausentes
products['product_name'] = products['product_name'].fillna('Unknown')

In [ ]:
# Conferindo
print(products['product_name'].isna().sum())

Nenhum nome ausente restante.

### Dias desde o pedido anterior (`days_since_prior_order`)

In [ ]:
# Valores ausentes por coluna
instacart_orders.isna().sum()

In [ ]:
# Ausentes entre clientes que NÃO estão no primeiro pedido
instacart_orders[instacart_orders['order_number'] > 1].isna().sum()

Entre clientes que **não** estão no primeiro pedido, não há nenhum ausente. Ou seja, os ausentes são todos de **primeiro pedido**: não existe compra anterior para medir o intervalo.

É uma ausência esperada, não um erro. **Mantive os valores vazios.** Preencher com zero, por exemplo, criaria clientes falsos que "voltaram no mesmo dia".

### Posição no carrinho (`add_to_cart_order`)

In [ ]:
# Valores ausentes por coluna
order_products.isna().sum()

In [ ]:
# Menor e maior posição no carrinho
print(order_products['add_to_cart_order'].min())
print(order_products['add_to_cart_order'].max())

A coluna vai de 1 a 64. Um teto redondo em 64 levanta suspeita: talvez o sistema pare de registrar a posição a partir do 65º item.

Testei essa hipótese olhando o tamanho dos pedidos que têm ausentes:

In [ ]:
# Pedidos com pelo menos um ausente em add_to_cart_order
miss_cart_order_ids = order_products[order_products['add_to_cart_order'].isna()]['order_id'].unique()

print('Valores ausentes:', order_products['add_to_cart_order'].isna().sum())
print('Pedidos afetados:', len(miss_cart_order_ids))

In [ ]:
# Quantos itens tem cada pedido afetado (contando todas as linhas do pedido)
itens_pedidos_afetados = (
    order_products[order_products['order_id'].isin(miss_cart_order_ids)]
    .groupby('order_id')['product_id']
    .count()
)

print('Menor pedido afetado:', itens_pedidos_afetados.min(), 'itens')

São **836 valores ausentes, em 70 pedidos**. E o menor desses pedidos tem **65 itens**.

Hipótese confirmada: o sistema só registra a posição até o 64º item. Preenchi os ausentes com **999**, um código claro de "acima de 64", e converti a coluna para inteiro.

In [ ]:
# Preenchendo com 999 e convertendo para inteiro
order_products['add_to_cart_order'] = order_products['add_to_cart_order'].fillna(999).astype(int)
order_products.info(show_counts=True)

Coluna sem ausentes e no tipo certo (`int64`).

## Resumo da preparação

| Problema | O que encontrei | Decisão |
|---|---|---|
| Pedidos duplicados | 15, todos na quarta às 2h | Removidos (falha de sistema) |
| Nomes de produto repetidos | 104 | Mantidos (IDs diferentes) |
| `product_name` ausente | 1.258, todos no corredor e departamento "missing" | Preenchidos com "Unknown" |
| `days_since_prior_order` ausente | 28.819, todos de primeiro pedido | Mantidos (ausência esperada) |
| `add_to_cart_order` ausente | 836, em 70 pedidos com mais de 64 itens | Preenchidos com 999 |

Base pronta para análise.

# Etapa 3. Análise

Com a base limpa, fui às perguntas de negócio.

### Os horários e dias fazem sentido?

In [ ]:
# Faixa de valores de hora e dia da semana
print('Hora:', instacart_orders['order_hour_of_day'].min(), 'a', instacart_orders['order_hour_of_day'].max())
print('Dia:', instacart_orders['order_dow'].min(), 'a', instacart_orders['order_dow'].max())

Horas de 0 a 23 e dias de 0 a 6 (0 = domingo). Tudo dentro do esperado.

### Em que horário as pessoas compram?

In [ ]:
orders_per_hour = instacart_orders['order_hour_of_day'].value_counts().sort_index()
print(orders_per_hour)

orders_per_hour.plot(
    kind='bar',
    rot=0,
    figsize=(10, 4),
    title='Quantidade de pedidos por hora do dia',
    xlabel='Hora do dia',
    ylabel='Quantidade de pedidos'
)
plt.show()

O movimento se concentra em horário comercial:
- **Das 10h às 16h** são mais de 38 mil pedidos por hora. O pico é **às 10h (40.578 pedidos)**.
- **De madrugada** (2h às 4h), cai para menos de mil por hora.
- A partir das 17h, o volume vai diminuindo.

**Na prática:** é a janela para concentrar push, campanha e promoção relâmpago.

### Em que dia da semana?

In [ ]:
dias = ['Dom', 'Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb']

orders_per_day = instacart_orders['order_dow'].value_counts().sort_index()
orders_per_day.index = dias
print(orders_per_day)

orders_per_day.plot(
    kind='bar',
    rot=0,
    title='Quantidade de pedidos por dia da semana',
    xlabel='Dia da semana',
    ylabel='Quantidade de pedidos'
)
plt.show()

**Domingo e segunda são os dias mais fortes**, com cerca de 84 mil e 82 mil pedidos. De terça a sábado o volume fica mais baixo e estável, entre 60 e 66 mil.

**Na prática:** reforço de estoque, equipe de entrega e verba de mídia no começo da semana.

### De quanto em quanto tempo os clientes voltam?

In [ ]:
days_wait = instacart_orders['days_since_prior_order'].value_counts().sort_index()
days_wait.index = days_wait.index.astype(int)

days_wait.plot(
    kind='bar',
    rot=0,
    figsize=(12, 4),
    title='Dias até o próximo pedido',
    xlabel='Dias desde o pedido anterior',
    ylabel='Quantidade de pedidos'
)
plt.show()

- O pico é em **7 dias**: muita gente faz compra semanal.
- Há picos menores em **14, 21 e 28 dias**, reforçando o ciclo semanal.
- A barra de **30 dias** é a maior de todas. Como 30 é o valor máximo da coluna, ela provavelmente junta todo mundo que demorou **30 dias ou mais**. Ou seja, não é um hábito mensal, e sim o grupo de clientes que esfriou.

**Na prática:** uma régua de reativação pode lembrar o cliente perto do 7º dia e ter uma ação específica para quem passou de 30.

### Quarta x sábado: o horário de compra muda?

In [ ]:
quarta_por_hora = instacart_orders[instacart_orders['order_dow'] == 3]['order_hour_of_day'].value_counts().sort_index()
sabado_por_hora = instacart_orders[instacart_orders['order_dow'] == 6]['order_hour_of_day'].value_counts().sort_index()

comparacao = pd.DataFrame({'Quarta': quarta_por_hora, 'Sábado': sabado_por_hora})
print(comparacao.sum())

comparacao.plot(
    kind='bar',
    rot=0,
    figsize=(12, 4),
    title='Pedidos por hora: quarta x sábado',
    xlabel='Hora do dia',
    ylabel='Quantidade de pedidos'
)
plt.show()

O volume é parecido, mas o formato da curva muda:
- **Quarta:** pico de manhã (10h–11h), leve queda no almoço (12h–13h) e novo pico às 15h.
- **Sábado:** a curva sobe sem interrupção e o pico fica **no começo da tarde (13h–14h)**.

Uma leitura possível: no dia útil, a rotina de trabalho interrompe a compra no almoço. No fim de semana, ela se espalha pelo dia.

### Quantos pedidos cada cliente faz?

In [ ]:
orders_per_user = instacart_orders.groupby('user_id')['order_id'].count()
distribution = orders_per_user.value_counts().sort_index()

print('Total de clientes:', len(orders_per_user))
print(distribution)

distribution.plot(
    kind='bar',
    rot=0,
    figsize=(10, 4),
    title='Número de pedidos por cliente',
    xlabel='Número de pedidos',
    ylabel='Número de clientes'
)
plt.show()

- **55.357 clientes fizeram só 1 pedido**, cerca de 35% de todos os clientes.
- A queda é rápida: 36.508 fizeram 2 pedidos e 21.547 fizeram 3.
- Pouquíssimos passam de 20 pedidos.

**Na prática:** o maior gargalo está entre o 1º e o 2º pedido. Cupom de segunda compra e régua de boas-vindas atacam exatamente esse ponto.

### Quais são os 20 produtos mais pedidos?

In [ ]:
top_20 = (
    order_products.groupby('product_id')
    .size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index(name='pedidos')
    .merge(products[['product_id', 'product_name']], on='product_id')
)

top_20[['product_id', 'product_name', 'pedidos']]

O ranking é dominado por **hortifrúti**: banana em 1º, depois banana orgânica, morango, espinafre e abacate. Dos 20 produtos, só o leite orgânico não é fruta ou verdura, e 15 são orgânicos.

**Na prática:** hortifrúti é o que traz o cliente de volta. É candidato natural a produto-isca em promoção.

### Quantos itens tem um pedido?

In [ ]:
items_per_order = order_products.groupby('order_id').size()
distribution = items_per_order.value_counts().sort_index()

print(items_per_order.describe())

distribution.head(40).plot(
    kind='bar',
    rot=0,
    figsize=(12, 4),
    title='Quantidade de itens por pedido (até 40 itens)',
    xlabel='Itens no pedido',
    ylabel='Número de pedidos'
)
plt.show()

- O tamanho mais comum é **5 itens** (31.923 pedidos).
- A maior parte dos pedidos é pequena, e o volume cai rápido conforme o carrinho cresce.
- Existe uma cauda longa: alguns pedidos passam de 100 itens (o maior tem 127).

O perfil é de compra de reposição, não de compra grande do mês.

### Quais produtos são mais recomprados?

In [ ]:
reordered_products = order_products[order_products['reordered'] == 1]

top_20_reordered = (
    reordered_products.groupby('product_id')
    .size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index(name='recompras')
    .merge(products[['product_id', 'product_name']], on='product_id')
)

top_20_reordered[['product_id', 'product_name', 'recompras']]

A lista é quase igual ao ranking geral: os mesmos itens de hortifrúti, mais leite orgânico e half & half. Ou seja, **o que mais vende é também o que mais se recompra**. São produtos de hábito.

### Qual a taxa de recompra de cada produto?

In [ ]:
reorder_rate = (
    order_products.groupby('product_id')['reordered']
    .mean()
    .mul(100)
    .round(1)
    .reset_index(name='taxa_recompra_%')
    .merge(products[['product_id', 'product_name']], on='product_id')
)

print(reorder_rate['taxa_recompra_%'].describe())
reorder_rate[['product_id', 'product_name', 'taxa_recompra_%']]

A taxa varia muito entre os produtos: tem produto com 0% e produto acima de 70%.

Isso separa dois tipos de item:
- **Produto de hábito** (taxa alta): bom para assinatura, lembrete de recompra e lista pronta.
- **Produto de experimentação** (taxa baixa): o cliente testa e não volta. Bom para descoberta e venda cruzada.

### Qual a taxa de recompra de cada cliente?

In [ ]:
reorder_rate_user = (
    order_products.merge(instacart_orders[['order_id', 'user_id']], on='order_id')
    .groupby('user_id')['reordered']
    .mean()
    .mul(100)
    .round(1)
)

print(reorder_rate_user.describe())

Também varia muito: há clientes com 0% (só compram coisa nova) e clientes acima de 90% (repetem quase tudo).

São perfis diferentes. Para quem repete, funcionam lembrete e lista pronta. Para quem varia, funcionam recomendação e novidade.

### O que vai primeiro para o carrinho?

In [ ]:
first_items = order_products[order_products['add_to_cart_order'] == 1]

top_20_first = (
    first_items.groupby('product_id')
    .size()
    .sort_values(ascending=False)
    .head(20)
    .reset_index(name='vezes_primeiro')
    .merge(products[['product_id', 'product_name']], on='product_id')
)

top_20_first[['product_id', 'product_name', 'vezes_primeiro']]

Banana, banana orgânica e **leite orgânico** lideram. E um detalhe aparece só aqui: **bebidas** como água, água com gás e refrigerante entram no top 20 dos primeiros itens, mas não no ranking geral.

Ou seja, o cliente abre o app já sabendo o que precisa repor. Esses itens são a porta de entrada do pedido: bom lugar para destaque na tela inicial e para sugestão de "compre junto".

# Conclusões

**O que a análise mostrou**
1. **Horário:** pedidos concentrados entre 10h e 16h, com pico às 10h. Madrugada quase parada.
2. **Dia:** domingo e segunda são os dias mais fortes.
3. **Recorrência:** o ciclo é semanal (pico em 7 dias), e a barra de 30+ dias mostra um grupo grande de clientes que esfriou.
4. **Retenção:** 35% dos clientes fizeram só 1 pedido. O maior gargalo é a segunda compra.
5. **Produtos:** hortifrúti domina vendas e recompra. Bebidas e leite aparecem como itens que abrem o carrinho.
6. **Perfis:** há clientes que repetem quase tudo e clientes que só compram novidade.

**Como eu usaria isso num negócio**
- Concentrar campanha e push entre 9h e 16h, com reforço no domingo e na segunda.
- Montar régua de reativação perto do 7º dia e uma ação específica para quem passou de 30 dias.
- Investir em incentivo de segunda compra.
- Usar hortifrúti como produto-isca e os itens de primeiro carrinho como vitrine de entrada.

**Preparação dos dados**

15 pedidos duplicados removidos (falha de sistema na quarta às 2h), 104 nomes repetidos mantidos com justificativa e valores ausentes tratados conforme a causa: 1.258 nomes de produto, 28.819 intervalos de primeiro pedido e 836 posições de carrinho em pedidos com mais de 64 itens.

**Limitações**
- A base é uma amostra reduzida da original. As contagens por cliente valem para os pedidos dessa amostra.
- As conclusões são descritivas. Para afirmar diferenças com segurança (ex.: quarta x sábado), o próximo passo seria um teste estatístico.